# 🕶️ Chrono S. Thompson — Suíte de Testes Interativa (Nodes & LangGraph)

Bem-vindo ao caderno de testes e desenvolvimento interativo do **Chrono S. Thompson: O Correspondente Temporal Gonzo**.

Este Jupyter Notebook permite testar **isoladamente** cada nó do fluxo do LangGraph e também executar o **grafo completo** de ponta a ponta.

---

## 🧭 Visão Geral do Grafo

```
[START] ──> fetch_events_node ──> rank_events_node ──> correlate_modern_node ──> write_article_node ──> [END]
```

### Estrutura dos Nós:
1. **`fetch_events_node`**: Consulta o MCP Server para obter eventos históricos ocorridos no dia selecionado (`target_date`).
2. **`rank_events_node`**: Filtra e ranqueia os eventos usando LLM com Structured Output (`RankedSelection`), gerando o hook editorial Gonzo.
3. **`correlate_modern_node`**: Pesquisa notícias/contexto atual (via Tavily ou síntese do LLM) e traça o paralelo temporal.
4. **`write_article_node`**: Redige a matéria em estilo Gonzo visceral e salva o arquivo `.md` no diretório de saída.


In [ ]:
# ==============================================================================
# 0. CONFIGURAÇÃO DO AMBIENTE E SUPORTE ASYNC NO JUPYTER
# ==============================================================================
import sys
import os
import asyncio
import logging
from pathlib import Path

# Adiciona a raiz do projeto e o diretório 'src' ao sys.path
project_root = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
src_dir = project_root / "src"

if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))
if str(src_dir) not in sys.path:
    sys.path.insert(0, str(src_dir))

# Permite chamadas asyncio aninhadas dentro de eventos do Jupyter Notebook
import nest_asyncio
nest_asyncio.apply()

# Configuração de logging visível no notebook
logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s [%(levelname)s] %(name)s: %(message)s",
    stream=sys.stdout
)
logger = logging.getLogger("JupyterTestRunner")

print("✅ Ambiente configurado com sucesso!")
print(f"📁 Project Root: {project_root}")


---
## 1. Estado Global Compartilhado (`ChronoState`)

O estado global trafega entre todos os nós do grafo. Vamos inspecionar os tipos de dados e os Schemas Pydantic.


In [ ]:
# ==============================================================================
# 1. INSPEÇÃO DAS ESTRUTURAS DE DADOS E ESTADO
# ==============================================================================
from src.chrono_s_thompson.core.state import ChronoState, HistoricalEvent, RankedSelection

# Exemplo de instância do estado inicial
sample_state: ChronoState = {
    "target_date": "08/27",
    "raw_events": [],
    "curated_story": None,
    "modern_context": None,
    "final_article": None,
    "published_path": None
}

print("📌 Estrutura das chaves do ChronoState:")
for key, value in sample_state.items():
    print(f"  - {key}: {type(value).__name__} = {value}")


---
## 2. Teste Isolado: Nó 1 — Fetcher Node (`fetch_events_node`)

O **Fetcher Node** comunica-se com o MCP Server (via `stdio`) para buscar fatos históricos do dia.


In [ ]:
# ==============================================================================
# 2. TESTANDO O FETCHER NODE ISOLADAMENTE
# ==============================================================================
from src.chrono_s_thompson.graph.nodes.fetcher import fetch_events_node

# Estado de entrada com a data alvo
test_fetch_state: ChronoState = {
    "target_date": "08/27",
    "raw_events": [],
    "curated_story": None,
    "modern_context": None,
    "final_article": None,
    "published_path": None
}

print("🚀 Executando fetch_events_node...")
fetch_result = await fetch_events_node(test_fetch_state)

raw_events = fetch_result.get("raw_events", [])
print(f"\n📊 Eventos retornados ({len(raw_events)} total):")

for idx, ev in enumerate(raw_events[:5], 1):
    year = ev.get("year", "N/A")
    title = ev.get("title", "")
    print(f"  {idx}. [{year}] {title}")


---
## 3. Teste Isolado: Nó 2 — Ranker Node (`rank_events_node`)

O **Ranker Node** consome a lista `raw_events` e utiliza o LLM com `with_structured_output(RankedSelection)` para escolher a melhor história com o tom Gonzo.


In [ ]:
# ==============================================================================
# 3. TESTANDO O RANKER NODE ISOLADAMENTE
# ==============================================================================
from src.chrono_s_thompson.graph.nodes.ranker import rank_events_node

# Se raw_events estiver vazio do teste anterior, criamos dados fictícios para teste isolado
if not raw_events:
    raw_events = [
        {
            "year": 1883,
            "title": "Erupção do Krakatoa atinge seu ápice violento.",
            "description": "A explosão vulcânica catastrófica gerou o som mais alto da história registrada.",
            "category": "História Geral"
        },
        {
            "year": 1910,
            "title": "Passagem do Cometa Halley causa pânico global.",
            "description": "Charlatões venderam pílulas anti-cometa enquanto o público aguardava o fim dos tempos.",
            "category": "Ciência e Sociedade"
        }
    ]

test_rank_state: ChronoState = {
    "target_date": "08/27",
    "raw_events": raw_events,
    "curated_story": None,
    "modern_context": None,
    "final_article": None,
    "published_path": None
}

print("🧠 Executando rank_events_node...")
rank_result = await rank_events_node(test_rank_state)

curated_story: RankedSelection = rank_result.get("curated_story")

if curated_story:
    print("\n🎯 Histórias Curadas e Selecionadas:")
    print(f"  - Ano: {curated_story.selected_event.year}")
    print(f"  - Evento: {curated_story.selected_event.title}")
    print(f"  - Hook Gonzo: {curated_story.gonzo_hook}")
    print(f"  - Tópico Moderno Sugerido: {curated_story.suggested_modern_topic}")
else:
    print("❌ Falha na curadoria do Ranker Node.")


---
## 4. Teste Isolado: Nó 3 — Correlator Node (`correlate_modern_node`)

O **Correlator Node** recebe a história curada e busca ou sintetiza conexões entre o evento passado e o cenário atual.


In [ ]:
# ==============================================================================
# 4. TESTANDO O CORRELATOR NODE ISOLADAMENTE
# ==============================================================================
from src.chrono_s_thompson.graph.nodes.correlator import correlate_modern_node

test_correlate_state: ChronoState = {
    "target_date": "08/27",
    "raw_events": raw_events,
    "curated_story": curated_story,
    "modern_context": None,
    "final_article": None,
    "published_path": None
}

print("🔗 Executando correlate_modern_node...")
correlate_result = await correlate_modern_node(test_correlate_state)

modern_context = correlate_result.get("modern_context")
print("\n🌐 Síntese do Paralelo Contemporâneo:\n")
print(modern_context)


---
## 5. Teste Isolado: Nó 4 — Writer Node (`write_article_node`)

O **Writer Node** encarna a persona do correspondente **Chrono S. Thompson** e redige a reportagem final em Markdown, salvando-a no diretório `storage/output/`.


In [ ]:
# ==============================================================================
# 5. TESTANDO O WRITER NODE ISOLADAMENTE
# ==============================================================================
from src.chrono_s_thompson.graph.nodes.writer import write_article_node
from IPython.display import display, Markdown

test_writer_state: ChronoState = {
    "target_date": "08/27",
    "raw_events": raw_events,
    "curated_story": curated_story,
    "modern_context": modern_context,
    "final_article": None,
    "published_path": None
}

print("✍️ Executando write_article_node...")
writer_result = await write_article_node(test_writer_state)

final_article = writer_result.get("final_article")
published_path = writer_result.get("published_path")

print(f"\n💾 Salvo em: {published_path}\n")
print("=" * 60)
display(Markdown(final_article if final_article else "Nenhum artigo gerado."))


---
## 6. Teste do Grafo Completo (`build_chrono_graph`)

Aqui compilamos o estado completo do LangGraph e executamos todo o pipeline de ponta a ponta com streaming de transições.


In [ ]:
# ==============================================================================
# 6. COMPILAÇÃO E EXECUÇÃO DO GRAFO COMPLETO
# ==============================================================================
from src.chrono_s_thompson.graph.builder import build_chrono_graph
from datetime import datetime

app = build_chrono_graph()

# Exibe a topologia do grafo
try:
    display(Markdown(f"```mermaid\n{app.get_graph().draw_mermaid()}\n```"))
except Exception:
    print("Topologia do Grafo:")
    print("START -> fetch_events -> rank_events -> correlate_modern -> write_article -> END")

today_str = datetime.now().strftime("%m/%d")

initial_state: ChronoState = {
    "target_date": today_str,
    "raw_events": [],
    "curated_story": None,
    "modern_context": None,
    "final_article": None,
    "published_path": None,
}

print(f"\n🚀 Executando Grafo Completo via astream() para a data {today_str}...\n")

async for event in app.astream(initial_state):
    for node_name, state_update in event.items():
        print(f"🔄 [Nó Concluído]: {node_name}")
        keys_updated = list(state_update.keys()) if isinstance(state_update, dict) else []
        print(f"   Chaves atualizadas no estado: {keys_updated}")


---
## 7. Playground de Injeção de Estado para Depuração

Utilize a célula abaixo para simular cenários específicos ou injetar dados customizados diretamente em qualquer nó.


In [ ]:
# ==============================================================================
# 7. PLAYGROUND DE INJEÇÃO DE ESTADO CUSTOMIZADO
# ==============================================================================
from src.chrono_s_thompson.core.state import HistoricalEvent, RankedSelection
from src.chrono_s_thompson.graph.nodes.writer import write_article_node

custom_event = HistoricalEvent(
    year=1969,
    title="Apollo 11 pousa na Lua.",
    description="Neil Armstrong e Buzz Aldrin tornam-se os primeiros humanos a caminhar em outro corpo celeste.",
    category="Exploração Espacial"
)

custom_story = RankedSelection(
    selected_event=custom_event,
    gonzo_hook="A humanidade abandona o lodo da Terra para flutuar em um deserto de poeira prateada sob o olhar de milionários e generais.",
    suggested_modern_topic="Privatização da corrida espacial e turismo orbital para bilionários."
)

custom_state: ChronoState = {
    "target_date": "07/20",
    "raw_events": [],
    "curated_story": custom_story,
    "modern_context": "Enquanto em 1969 o espaço era um teatro de guerra fria estatal, hoje foguetes em formato de charuto levam bilionários para voos de 5 minutos alimentados por narcisismo e capital de risco.",
    "final_article": None,
    "published_path": None
}

print("🧪 Testando o Writer Node com Estado Customizado...")
result = await write_article_node(custom_state)
display(Markdown(result["final_article"]))
